# Lab 3: Planowanie z wykorzystaniem języka PDDL

## Wprowadzenie teoretyczne

PDDL (Planning Domain Definition Language) to formalny język do opisu problemów planowania w AI, zbudowany na notacji prefiksowej (S-wyrażenia). Definicja PDDL składa się z:

- **Domeny** (`domain.pddl`): rodzaje obiektów, predykaty, akcje i ich efekty
- **Problemu** (`problem.pddl`): obiekty, stan początkowy, cel planowania

Poniżej zaimplementujemy prosty generator PDDL w Pythonie, a następnie użyjemy go do modeli z zadań.

## Implementacja generatora PDDL w Pythonie

Zdefiniujemy klasy `PDDLDomain` i `PDDLProblem` do budowania plików PDDL programowo.

In [1]:
class PDDLDomain:
    def __init__(self, name, requirements=None, types=None, predicates=None, actions=None):
        self.name = name
        self.requirements = requirements or []
        self.types = types or []
        self.predicates = predicates or []  # list of strings
        self.actions = actions or []        # list of PDDLAction

    def to_pddl(self):
        parts = [f"(define (domain {self.name})"]
        if self.requirements:
            reqs = ' '.join(self.requirements)
            parts.append(f"  (:requirements {reqs})")
        if self.types:
            types_str = ' '.join(self.types)
            parts.append(f"  (:types {types_str})")
        if self.predicates:
            parts.append("  (:predicates")
            for pred in self.predicates:
                parts.append(f"    {pred}")
            parts.append("  )")
        for action in self.actions:
            parts.append(action.to_pddl())
        parts.append(")")
        return '\n'.join(parts)

class PDDLAction:
    def __init__(self, name, parameters, precondition, effect):
        self.name = name
        self.parameters = parameters  # list of strings like '?r - robot'
        self.precondition = precondition  # string
        self.effect = effect              # string

    def to_pddl(self):
        params = ' '.join(self.parameters)
        return (f"  (:action {self.name}\n"
                f"    :parameters ({params})\n"
                f"    :precondition {self.precondition}\n"
                f"    :effect {self.effect}\n"
                f"  )")

class PDDLProblem:
    def __init__(self, name, domain, objects=None, init=None, goal=None):
        self.name = name
        self.domain = domain
        self.objects = objects or {}  # dict type->list of names
        self.init = init or []        # list of strings
        self.goal = goal              # string

    def to_pddl(self):
        parts = [f"(define (problem {self.name})",
                 f"  (:domain {self.domain})"]
        if self.objects:
            parts.append("  (:objects")
            for t, objs in self.objects.items():
                line = ' '.join(objs) + ' - ' + t
                parts.append(f"    {line}")
            parts.append("  )")
        if self.init:
            parts.append("  (:init")
            for fact in self.init:
                parts.append(f"    {fact}")
            parts.append("  )")
        if self.goal:
            parts.append(f"  (:goal {self.goal})")
        parts.append(")")
        return '\n'.join(parts)

## Zadanie 1: Transport paczek

Modelujemy jednego agenta transportującego paczki pomiędzy lokacjami. Wykorzystamy rozszerzenia `:strips`, `:typing`, `:negative-preconditions` i `:numeric-fluents` dla kosztów.


In [2]:
# Definicja domeny dla zadania 1
domain1 = PDDLDomain(
    name='transport-paczek',
    requirements=[':strips', ':typing', ':negative-preconditions', ':numeric-fluents'],
    types=['robot', 'package', 'location'],
    predicates=[
        '(at ?r - robot ?l - location)',
        '(at-packet ?p - package ?l - location)',
        '(connected ?from ?to - location)'
    ],
    actions=[
        PDDLAction(
            name='load',
            parameters=['?r - robot', '?p - package', '?l - location'],
            precondition='(and (at ?r ?l) (at-packet ?p ?l))',
            effect='(and (not (at-packet ?p ?l)) (loaded ?p ?r))'
        ),
        PDDLAction(
            name='unload',
            parameters=['?r - robot', '?p - package', '?l - location'],
            precondition='(loaded ?p ?r)',
            effect='(and (at-packet ?p ?l) (not (loaded ?p ?r)))'
        ),
        PDDLAction(
            name='move',
            parameters=['?r - robot', '?from - location', '?to - location'],
            precondition='(and (at ?r ?from) (connected ?from ?to))',
            effect='(and (not (at ?r ?from)) (at ?r ?to))'
        )
    ]
)

problem1 = PDDLProblem(
    name='transport1',
    domain='transport-paczek',
    objects={
        'location': ['loc1', 'loc2', 'loc3'],
        'robot': ['rob1'],
        'package': ['pkg1', 'pkg2']
    },
    init=[
        '(at rob1 loc1)',
        '(at-packet pkg1 loc1)',
        '(at-packet pkg2 loc2)',
        '(connected loc1 loc2)',
        '(connected loc2 loc3)',
        '(connected loc3 loc1)'
    ],
    goal='(and (at-packet pkg1 loc3) (at-packet pkg2 loc1))'
)

# Zapis do plików
with open('domain1.pddl', 'w') as f:
    f.write(domain1.to_pddl())
with open('problem1.pddl', 'w') as f:
    f.write(problem1.to_pddl())

print('Zapisano domain1.pddl i problem1.pddl')

Zapisano domain1.pddl i problem1.pddl


## Zadanie 2: Sprzątający robot

Klasyczny problem odwiedzenia i posprzątania wszystkich pokoi.

In [4]:
# Definicja domeny dla zadania 2
domain2 = PDDLDomain(
    name='robot-cleaner',
    requirements=[':strips', ':typing'],
    types=['robot', 'room'],
    predicates=[
        '(at ?r - robot ?p - room)',
        '(dirty ?p - room)',
        '(clean ?p - room)'
    ],
    actions=[
        PDDLAction(
            name='move',
            parameters=['?r - robot', '?from - room', '?to - room'],
            precondition='(at ?r ?from)',
            effect='(and (not (at ?r ?from)) (at ?r ?to))'
        ),
        PDDLAction(
            name='clean',
            parameters=['?r - robot', '?p - room'],
            precondition='(and (at ?r ?p) (dirty ?p))',
            effect='(and (clean ?p) (not (dirty ?p)))'
        )
    ]
)

problem2 = PDDLProblem(
    name='clean1',
    domain='robot-cleaner',
    objects={'room': ['pokoj1', 'pokoj2', 'pokoj3'], 'robot': ['robo']},
    init=[
        '(at robo pokoj1)',
        '(dirty pokoj1)',
        '(dirty pokoj2)',
        '(dirty pokoj3)'
    ],
    goal='(and (clean pokoj1) (clean pokoj2) (clean pokoj3))'
)

with open('domain2.pddl', 'w') as f:
    f.write(domain2.to_pddl())
with open('problem2.pddl', 'w') as f:
    f.write(problem2.to_pddl())

print('Zapisano domain2.pddl i problem2.pddl')


Zapisano domain2.pddl i problem2.pddl


## Zadanie 3: Przenoszenie piłek wieloramiennym robotem

Model z dwoma ramionami i czterema piłkami.

In [5]:
# Definicja domeny dla zadania 3
domain3 = PDDLDomain(
    name='ball-moving-robot',
    requirements=[':strips', ':typing'],
    types=['robot', 'room', 'ball', 'arm'],
    predicates=[
        '(at ?r - robot ?rm - room)',
        '(inroom ?b - ball ?rm - room)',
        '(holding ?a - arm ?b - ball)',
        '(arm-empty ?a - arm)'
    ],
    actions=[
        PDDLAction(
            name='move',
            parameters=['?r - robot', '?from - room', '?to - room'],
            precondition='(at ?r ?from)',
            effect='(and (not (at ?r ?from)) (at ?r ?to))'
        ),
        PDDLAction(
            name='pick-up',
            parameters=['?r - robot', '?a - arm', '?b - ball', '?rm - room'],
            precondition='(and (at ?r ?rm) (inroom ?b ?rm) (arm-empty ?a))',
            effect='(and (holding ?a ?b) (not (arm-empty ?a)) (not (inroom ?b ?rm)))'
        ),
        PDDLAction(
            name='put-down',
            parameters=['?r - robot', '?a - arm', '?b - ball', '?rm - room'],
            precondition='(and (at ?r ?rm) (holding ?a ?b))',
            effect='(and (inroom ?b ?rm) (arm-empty ?a) (not (holding ?a ?b)))'
        )
    ]
)

problem3 = PDDLProblem(
    name='move-balls',
    domain='ball-moving-robot',
    objects={
        'room': ['room1', 'room2'],
        'robot': ['robot'],
        'ball': ['ball1', 'ball2', 'ball3', 'ball4'],
        'arm': ['arm1', 'arm2']
    },
    init=[
        '(at robot room1)',
        '(inroom ball1 room1)',
        '(inroom ball2 room1)',
        '(inroom ball3 room1)',
        '(inroom ball4 room1)',
        '(arm-empty arm1)',
        '(arm-empty arm2)'
    ],
    goal='(and (inroom ball1 room2) (inroom ball2 room2) (inroom ball3 room2) (inroom ball4 room2))'
)

with open('domain3.pddl', 'w') as f:
    f.write(domain3.to_pddl())
with open('problem3.pddl', 'w') as f:
    f.write(problem3.to_pddl())

print('Zapisano domain3.pddl i problem3.pddl')

Zapisano domain3.pddl i problem3.pddl
